# اليوم 2 — Staging وSilver وdbt — Lab 03

أمل خوتاني — Amal Khotani.

ضمن **هندسة البيانات الحديثة لأنظمة الذكاء الاصطناعي — Modern Data Engineering for AI Systems (SDA-DSC-214)** لدى [أكاديمية سدايا](https://github.com/SDAIAAcademy). #SDAIAAcademy

مواد الدورة ودوالها: **ميعاد المري — Meaad Al-Marri**. البيانات اصطناعية.

هذه نسخة منظمة من قسم اليوم 2 في الدفتر المرفوع `amal_khotani (3).ipynb`؛ حُفظت شيفرة الخلايا الأصلية ومخرجاتها وأرقام تنفيذها كما وردت. لم يُعد تشغيل اللابات أثناء فصل الدفاتر، ولم يُختبر Run all في جلسة نظيفة. أرقام التنفيذ تعود إلى جلسات مختلفة.

قبل إعادة التشغيل، اتبع `docs/SETUP.md` في المستودع: Python 3.11، Java 17، PySpark 3.5.8، Delta Spark 3.3.3 وPy4J 0.10.9.9. يجب أن يكون مستودع المقرر وبياناته ومساحة العمل المطلوبة متاحين؛ وجود المخرجات المحفوظة وحده لا يجهز بيئة التشغيل.

ابدأ بعد إكمال اليوم 1؛ في جلسة جديدة استعد أرشيف ذلك اليوم إلى جذر المستودع وأعد تهيئة البيئة، ثم شغّل خلية Continue your project.



Day 2 · ELT and Silver

In [25]:
from pathlib import Path
import json, sys
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'course.json').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Open the notebook inside the complete course repository; see docs/SETUP.md.')
sys.path.insert(0, str(ROOT / 'src'))
SOURCE = ROOT / 'data/masar-small-v1'
from masar.workspace import require_fixed_dataset, completed_bronze_workspace
from masar.runtime import require_environment, start_spark
from masar.native_contracts import validate_stage_result
require_fixed_dataset(SOURCE)
require_environment()
WORK = completed_bronze_workspace(ROOT)
print('Continue workspace:', WORK.relative_to(ROOT))

Continue workspace: outputs/day01_bronze_xgnqdyt3


03a · Prepare typed staging

In [26]:
from masar.silver import run_staging_lab
spark = start_spark(WORK, kafka=False)
try:
    result = run_staging_lab(spark, SOURCE, WORK)
    validate_stage_result('lab03a_staging', result)
    print(json.dumps({'scope': result['scope'], 'checks': result['checks']}, indent=2))
    # Observed learning output
    print('Observed staging row counts:', result['counts'])
    preview = WORK / ('mini_lakehouse/staging/day02_' + result['run_id'] + '/stg_trips')
    spark.read.format('delta').load(str(preview)).select('trip_id', 'city', 'fare_sar').orderBy('trip_id').show(5, truncate=False)
finally:
    spark.stop()

{
  "scope": "DAY02_STAGING_ENGINE",
  "checks": {
    "counts_verified": true,
    "typed_values_match_source_oracle": true,
    "drivers_unique_and_join_safe": true,
    "gps_valid": true,
    "delta_readback": true
  }
}
Observed staging row counts: {'stg_trips': 144, 'stg_drivers': 6, 'stg_gps': 216}
+---------+------+--------+
|trip_id  |city  |fare_sar|
+---------+------+--------+
|SYN_T0001|Riyadh|18.00   |
|SYN_T0001|Riyadh|18.00   |
|SYN_T0002|Riyadh|19.25   |
|SYN_T0002|Riyadh|19.25   |
|SYN_T0003|Riyadh|20.50   |
+---------+------+--------+
only showing top 5 rows



03b · Build incremental Silver

In [27]:
from masar.silver import run_incremental_lab
spark = start_spark(WORK, kafka=False)
try:
    result = run_incremental_lab(spark, SOURCE, WORK)
    validate_stage_result('lab03b_silver', result)
    print(json.dumps({'scope': result['scope'], 'checks': result['checks']}, indent=2))
    # Observed learning output
    spark.read.format('delta').load(str(WORK/'mini_lakehouse/silver/trips')).select('trip_id', 'city', 'fare_sar').orderBy('trip_id').show(5, truncate=False)
finally:
    spark.stop()

{
  "scope": "DAY02_SILVER_ENGINE",
  "checks": {
    "all_scenarios_match_independent_oracle": true,
    "business_keys_unique": true,
    "replay_preserves_business_content": true,
    "late_rows_retained": true,
    "actual_delta_files": true
  }
}
+-----------+------+--------+
|trip_id    |city  |fare_sar|
+-----------+------+--------+
|SYN_LATE001|Riyadh|25.00   |
|SYN_LATE002|Jeddah|27.00   |
|SYN_LATE003|Dammam|29.00   |
|SYN_T0001  |Riyadh|18.00   |
|SYN_T0002  |Riyadh|19.25   |
+-----------+------+--------+
only showing top 5 rows



Lab 03 · Run the dbt models

In [29]:
import sys, subprocess, json

subprocess.run([
    sys.executable, "-m", "pip", "install",
    "-r", str(ROOT / "requirements-dbt.txt")
], check=True)

from masar.dbt_lab import run_dbt_lab

dbt_report, dbt_path = run_dbt_lab(ROOT)

print(json.dumps({
    "status": dbt_report["status"],
    "phases_completed": len(dbt_report["phases"]),
    "report": str(dbt_path.relative_to(ROOT)),
    "error": dbt_report.get("error")
}, indent=2))

assert dbt_report["status"] == "PASSED_DBT_NATIVE", dbt_report.get("error")

for phase in dbt_report["phases"]:
    print(phase["phase"], "rows:", phase["rows"],
          "fare SAR:", phase["total_fare_sar"])

print("Catalog evidence:", dbt_report["commands"][-1])

{
  "status": "PASSED_DBT_NATIVE",
  "phases_completed": 4,
  "report": "outputs/dbt_validation_rbj8s_xq/reports/dbt_attempt.json",
  "error": null
}
base rows: 72 fare SAR: 1794.60
rerun rows: 72 fare SAR: 1794.60
late rows: 75 fare SAR: 1875.60
late_replay rows: 75 fare SAR: 1875.60
Catalog evidence: {'catalog_sha256': 'e3c0ab0c53548621b0d4efd0da7a2f81703966c1e0bc2534889e367230314c3e', 'models_documented': 6, 'sources_documented': 3, 'metadata_method': 'native DESCRIBE TABLE EXTENDED', 'phase': 'documentation', 'command': ['docs', 'generate'], 'target': 'dbt/commands/documentation/target'}


Review and save

In [31]:
from pathlib import Path
import zipfile
pointer = ROOT / 'outputs/day01_bronze_success.json'
archive = ROOT / 'outputs/day02_handoff.zip'
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as bundle:
    dbt_workspace = dbt_path.parent.parent
    for p in sorted(dbt_workspace.rglob('*')):
        if p.is_file():
            bundle.write(p, p.relative_to(ROOT).as_posix())
    bundle.write(pointer, pointer.relative_to(ROOT).as_posix())
    for path in sorted(WORK.rglob('*')):
        if path.is_file():
            bundle.write(path, path.relative_to(ROOT).as_posix())
with zipfile.ZipFile(archive) as bundle:
    assert bundle.testzip() is None
print('Retain the notebook outputs, notes and', archive.relative_to(ROOT))

Retain the notebook outputs, notes and outputs/day02_handoff.zip


## ملحق إعادة تشغيل dbt وحفظ أدلته

بعد فقد أرشيف dbt القديم، أُعيد تشغيل مشغّل المقرر على نسخ مستقلة من لقطات Bronze الأصلية. توجد خلايا الاستعادة وإعادة التشغيل بمخرجاتها الفعلية في [DBT_RECOVERY.ipynb](DBT_RECOVERY.ipynb).

التقرير الجديد: `outputs/dbt_validation_tedk3wzl/reports/dbt_attempt.json`، وحالته `PASSED_DBT_NATIVE`. الأرشيف المحفوظ: `dbt_evidence_83b24dcb.zip`، ويشمل التوثيق المولد لـ6 نماذج و3 مصادر. لا يُنسب هذا التقرير الجديد إلى تشغيل dbt القديم المعروض أعلاه، ولا يستبدل أرشيف اليوم الخامس. تفاصيل التحقق في [LAB03_NOTES.md](../LAB03_NOTES.md) و[EVIDENCE_INDEX.md](../EVIDENCE_INDEX.md).
